# MLP Experiment — Phase 0 Analysis

Torch MLP (4-16-16-3), iris, 5-fold, 10 seeds, `tc=2`, `feat=2`.
Each fold fits **one** model attacked by **both** HopSkipJump (black-box) and DeepFool
(white-box), so the attack contrast shares identical weights.

Spread is inverse-transformed back to raw iris units, then normalised **per seed** to that
seed's own clean baseline. **1.0 = no effect.**

Written narrative: `FINDINGS_phase0.md` · contract: `PLAN.md` · text tables: `analyze.py`

In [ ]:
from pathlib import Path
import numpy as np, polars as pl
import matplotlib.pyplot as plt

HERE = Path.cwd() if Path.cwd().name == 'mlp_experiment' else Path.cwd() / 'mlp_experiment'
PLOTS = HERE / 'plots'; PLOTS.mkdir(exist_ok=True)
DF = pl.read_parquet(HERE / 'results.parquet')

# Tufte-ish house style, matching generate_figures_v2.py (the repo's preferred figures)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.25, 'legend.frameon': False})
C = {'hsj': '#1e5eff', 'deepfool': '#e8590c'}          # blue = black-box, orange = white-box
LS = {'before_split': '--', 'train_only': '-'}

def ci95(v):
    """mean, half-width of the 95% CI (NaN-safe)."""
    v = np.asarray([x for x in v if np.isfinite(x)], float)
    if len(v) < 2:
        return (float(v[0]) if len(v) else np.nan), np.nan
    return float(v.mean()), float(1.96 * v.std(ddof=1) / np.sqrt(len(v)))

def normalise(df, baseline):
    """norm = mean_dist / that (attack, seed)'s clean baseline — paired, so 10 seeds suffice."""
    b = baseline.group_by(['attack', 'seed']).agg(pl.col('mean_dist').mean().alias('_b'))
    return df.join(b, on=['attack', 'seed'], how='left').with_columns(
        (pl.col('mean_dist') / pl.col('_b')).alias('norm'))

CG = DF.filter(pl.col('defect') == 'coverage_gap')
OL = DF.filter(pl.col('defect') == 'outlier')
CG_N = normalise(CG.filter(pl.col('bias') > 0), CG.filter(pl.col('bias') == 0.0))
OL_N = normalise(OL.filter(pl.col('n_out') > 0), OL.filter(pl.col('n_out') == 0))
print(f'{len(DF)} rows / {len(DF)//2} cells, {int(DF["mean_dist"].is_nan().sum())} NaN')

## F1 — the headline: "accuracy stays flat" is a test-set artifact

Same defect, same model, same weights. Only **when** the deletion happens changes.

Under `before_split` the deleted region leaves the *test* set too, so the model is scored only
on the easy remainder — accuracy does not stay flat, it **rises to 0.999** and minority recall
reaches **1.000**. Under the correct `train_only` protocol both collapse.

This is the controlled version of the 2026-08-11 imbalance finding, which changed protocol *and*
tree depth together. An MLP has no depth knob, so the protocol is isolated.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True)
biases = sorted(b for b in CG['bias'].unique() if b > 0)

for ax, metric, title in [(axes[0], 'vacc', 'Test accuracy'),
                          (axes[1], 'min_recall', 'Minority-class recall (virginica)')]:
    for proto in ['before_split', 'train_only']:
        sub = CG.filter((pl.col('protocol') == proto) & (pl.col('bias') > 0) &
                        (pl.col('attack') == 'hsj'))          # metric is attack-independent
        m = [ci95(sub.filter(pl.col('bias') == b)[metric].to_list()) for b in biases]
        mu, hw = np.array([x[0] for x in m]), np.array([x[1] for x in m])
        ax.plot(biases, mu, LS[proto], marker='o', ms=4,
                color='#c92a2a' if proto == 'train_only' else '#868e96',
                label='train_only (correct)' if proto == 'train_only' else 'before_split (grids 1-3)')
        ax.fill_between(biases, mu - hw, mu + hw, alpha=0.15,
                        color='#c92a2a' if proto == 'train_only' else '#868e96')
    ax.set_title(title); ax.set_xlabel('bias (fraction of class deleted)'); ax.set_ylim(0, 1.05)
axes[0].set_ylabel('score'); axes[0].legend(loc='lower left')
fig.suptitle('The defect is only invisible to accuracy when you also delete it from the test set',
             y=1.02, fontsize=11)
fig.tight_layout(); fig.savefig(PLOTS / 'f1_protocol_accuracy.png', bbox_inches='tight')
plt.show()

## F2 / F3 — the spread signal itself, and the bias-0.9 HSJ drop

The signal does **not** depend on the confound: under `train_only` both attacks still rise
monotonically (HSJ d = +2.81).

Watch the dashed blue line: HSJ/`before_split` reproduces the **non-monotonic drop at bias 0.9**
seen on Tree+HSJ, and it **disappears** under `train_only` (solid blue) — confirming test-set
depletion as the mechanism, which the repo had flagged as asserted but unverified.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for attack in ['hsj', 'deepfool']:
    for proto in ['before_split', 'train_only']:
        sub = CG_N.filter((pl.col('attack') == attack) & (pl.col('protocol') == proto))
        m = [ci95(sub.filter(pl.col('bias') == b)['norm'].to_list()) for b in biases]
        mu, hw = np.array([x[0] for x in m]), np.array([x[1] for x in m])
        ax.errorbar(biases, mu, yerr=hw, fmt=LS[proto], marker='o', ms=4, capsize=3,
                    color=C[attack], alpha=1.0 if proto == 'train_only' else 0.55,
                    label=f'{attack} / {proto}')
ax.axhline(1.0, color='k', lw=0.8, ls=':')
ax.annotate('no effect', xy=(0.9, 1.0), xytext=(0.78, 0.955), fontsize=8, color='k')
ax.annotate('HSJ drop:\ntest set depleted', xy=(0.9, 1.02), xytext=(0.62, 1.24), fontsize=8,
            color=C['hsj'], ha='center',
            arrowprops=dict(arrowstyle='->', color=C['hsj'], lw=0.8))
ax.set_xlabel('bias (fraction of class deleted)')
ax.set_ylabel('normalised adversarial spread  (1.0 = clean baseline)')
ax.set_title('Coverage gap: the geometric signal survives a global smooth boundary')
ax.legend(loc='upper left', fontsize=8)
fig.tight_layout(); fig.savefig(PLOTS / 'f2_coverage_gap_spread.png', bbox_inches='tight')
plt.show()

## F4 — H6 refuted: a global boundary does *not* see outward outliers

Pre-registered prediction: an MLP's globally-parameterised boundary should be dragged by a far
same-labeled point, so `outward` should fire where every local model is null.

It doesn't — and neither does `toward`. So rbf **locality** does not explain the SVM null, and
the tree's 1.25–1.36× now looks specific to *axis-aligned partitioning under white-box attack*.

In [ ]:
# Reference values from Thread A / B findings, same strong cell (k=8, n_out=10)
REF = [('tree d3\n(DTA, WB)', 1.36, 1.01), ('tree d10\n(DTA, WB)', 1.25, 1.00),
       ('rbf-SVM\n(HSJ)', 0.98, 1.05), ('RandomForest\n(HSJ)', 0.92, 1.04)]
strong = OL_N.filter((pl.col('k') == 8.0) & (pl.col('n_out') == 10))
mlp = []
for a, lbl in [('hsj', 'MLP\n(HSJ)'), ('deepfool', 'MLP\n(DeepFool, WB)')]:
    vals = {d: ci95(strong.filter((pl.col('attack') == a) & (pl.col('direction') == d))['norm'].to_list())
            for d in ['toward', 'outward']}
    mlp.append((lbl, vals['toward'], vals['outward']))

labels = [r[0] for r in REF] + [m[0] for m in mlp]
tw = np.array([r[1] for r in REF] + [m[1][0] for m in mlp])
ow = np.array([r[2] for r in REF] + [m[2][0] for m in mlp])
tw_e = np.array([m[1][1] for m in mlp]); ow_e = np.array([m[2][1] for m in mlp])

n = len(REF); x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(9.5, 4.4))
# The reference bars are POINT ESTIMATES quoted from the prior findings docs, which do not
# publish CIs. Drawn faded and WITHOUT error bars, so they are not read as more precise than
# this thread's 10-seed measurements -- zero-length caps would imply exactly that.
ax.bar(x[:n] - w/2, tw[:n], w, color='#2f9e44', alpha=0.45, label='toward')
ax.bar(x[:n] + w/2, ow[:n], w, color='#adb5bd', alpha=0.45, label='outward')
ax.bar(x[n:] - w/2, tw[n:], w, yerr=tw_e, capsize=3, color='#2f9e44')
ax.bar(x[n:] + w/2, ow[n:], w, yerr=ow_e, capsize=3, color='#adb5bd')
ax.axhline(1.0, color='k', lw=0.8, ls=':')
ax.axvline(n - 0.5, color='k', lw=0.8, alpha=0.3)
ax.text(n - 0.45, 1.44, 'this thread (10 seeds, 95% CI)', fontsize=9, style='italic')
ax.text(-0.4, 1.44, 'prior findings (point estimates, no CIs published)',
        fontsize=9, style='italic', color='#666')

# n_out=10 is only 25% of the 40-point class. The MLP's `toward` response switches on at 50%
# and holds ~1.10x to 150% (see the ceiling figure) -- reading "the MLP is null" off this panel
# alone was the error corrected in FINDINGS_phase0.md F4. Mark it rather than hide it.
hi = OL_N.filter((pl.col('k') == 8.0) & (pl.col('n_out') == 40) &
                 (pl.col('attack') == 'deepfool') & (pl.col('direction') == 'toward'))
m_hi, h_hi = ci95(hi['norm'].to_list())
ax.errorbar([len(labels) - 1 - w/2], [m_hi], yerr=[h_hi], fmt='D', ms=7, capsize=4,
            color='#087f5b', zorder=5)
ax.annotate(f'same model at 100% dose:\n{m_hi:.2f}x [{m_hi-h_hi:.2f},{m_hi+h_hi:.2f}]',
            xy=(len(labels) - 1 - w/2, m_hi), xytext=(len(labels) - 2.5, 1.30),
            fontsize=8, color='#087f5b',
            arrowprops=dict(arrowstyle='->', color='#087f5b', lw=0.9))
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8.5)
ax.set_ylabel('normalised spread  (1.0 = no effect)'); ax.set_ylim(0.8, 1.5)
ax.set_title('Outlier at MATCHED dose (k=8, n_out=10 = 25% of class) — the MLP needs twice this')
ax.legend(loc='upper right')
fig.tight_layout(); fig.savefig(PLOTS / 'f4_outlier_model_comparison.png', bbox_inches='tight')
plt.show()

## H7 — the response is real, but it is to contamination, not to outliers

Trees only react at `k >= 6` (~6 sigma). The open question was whether a more sensitive model
reacts at *realistic* 2–3 sigma. The answer is subtler than yes or no.

`toward` aims the injection at the nearest other class. For `tc=2` the virginica→versicolor
centroid gap is **1.62 cm**, while one `k` step is only **0.442 cm** — so the injected point
crosses into versicolor territory at about **k ≈ 3.7**. Below that it is not an outlier at all,
it is *contamination*: points labelled virginica sitting inside the versicolor cloud.

DeepFool responds strongly there — by **compressing** (0.85× at `k=2`, dose-dependent), because
the contamination anchors the boundary in the dense contested band and every minimum-norm
adversarial piles into it. HSJ misses it entirely: random init plus a Monte-Carlo direction
estimate smear the landing points.

Above the frontier, where the injection is a genuine far-field outlier, **both attacks are null**.

*(An earlier version of this notebook shaded k=2–3 as "realistic outliers" and titled the figure
"no dose-response at any distance". Both were wrong — that band is the contamination zone, and it
is where the thread's only clean dose-response lives.)*

In [ ]:
from sklearn.datasets import load_iris
Xi, yi = load_iris(return_X_y=True)
cmu = {c: Xi[yi == c].mean(axis=0) for c in (0, 1, 2)}
s_scale = float(Xi[yi == 2].std(axis=0).mean())          # the sigma that k multiplies
gap = float(np.linalg.norm(cmu[1] - cmu[2]))             # virginica -> versicolor centroid
k_front = gap / s_scale                                  # k at which the point reaches versicolor
print(f'sigma={s_scale:.3f} cm, centroid gap={gap:.2f} cm -> contamination frontier k={k_front:.2f}')

ks = sorted(OL_N['k'].unique())
LBL = {'hsj': 'HSJ (black-box)', 'deepfool': 'DeepFool (white-box)'}
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), sharey=True)
for ax, d in zip(axes, ['toward', 'outward']):
    for attack in ['hsj', 'deepfool']:
        sub = OL_N.filter((pl.col('attack') == attack) & (pl.col('direction') == d) &
                          (pl.col('n_out') == 10))
        m = [ci95(sub.filter(pl.col('k') == k)['norm'].to_list()) for k in ks]
        mu, hw = np.array([x[0] for x in m]), np.array([x[1] for x in m])
        ax.errorbar(ks, mu, yerr=hw, marker='o', ms=4, capsize=3, color=C[attack], label=LBL[attack])
    ax.axhline(1.0, color='k', lw=0.8, ls=':')
    if d == 'toward':
        # Below the frontier the injected point lands INSIDE versicolor, so it is CONTAMINATION,
        # not an outlier. Shading this -- rather than a "realistic outliers" band -- is the whole
        # point: the only response in the outlier grid lives on the wrong side of this line.
        ax.axvspan(min(ks), k_front, color='#e8590c', alpha=0.10)
        ax.axvline(k_front, color='#e8590c', lw=1.0, ls='--')
        ax.text((min(ks) + k_front) / 2, 0.805, 'CONTAMINATION\nlands inside versicolor',
                fontsize=7.5, ha='center', color='#c1440e')
        ax.text(k_front + 0.15, 0.805, 'true far-field\noutlier', fontsize=7.5, color='#666')
        ax.annotate(f'versicolor centroid: k={k_front:.1f}', xy=(k_front, 1.10),
                    xytext=(k_front + 0.45, 1.15), fontsize=7.5, color='#c1440e',
                    arrowprops=dict(arrowstyle='->', color='#c1440e', lw=0.8))
    ax.set_title(f'direction = {d}'); ax.set_xlabel('k  (distance in class sigma)')
    sec = ax.secondary_xaxis('top', functions=(lambda v: v * s_scale, lambda v: v / s_scale))
    sec.set_xlabel('distance from class centroid (cm)', fontsize=8); sec.tick_params(labelsize=7.5)
axes[0].set_ylabel('normalised spread'); axes[0].set_ylim(0.78, 1.20)
axes[0].legend(loc='lower right', fontsize=8)
fig.suptitle('DeepFool detects near-field CONTAMINATION by compressing; '
             'both attacks are null on true outliers  (n_out=10)', y=1.12, fontsize=10.5)
fig.tight_layout(); fig.savefig(PLOTS / 'h7_outlier_dose_response.png', bbox_inches='tight')
plt.show()

## Ceiling — does raising the dose rescue the null?

The train fold holds **40 points per class**, so `n_out=40` doubles the target class and `n_out=60`
is 150% of it. Thread A found three distinct shapes at the ceiling
([`FINDINGS_outlier.md`](../model_family_experiment/FINDINGS_outlier.md) F3):

- **tree + DTA** — rises, plateaus ~1.25–1.36%, then **collapses** at 100% (the outliers stop
  being anomalies and form their own blob, so the tree stops stratifying them);
- **rbf-SVM + HSJ** — null at low dose but **rises to 1.09×** at the ceiling, and uniquely its
  **test accuracy degrades** 0.97 → 0.89;
- **RandomForest + HSJ** — flat null at every dose.

The MLP is the *other* global model, so the SVM shape is the one to watch for. Note the reading
rule: spread rising **together with** accuracy degrading is contamination, not a hidden defect —
it fails the accuracy-blindness test that motivates the whole project.

In [ ]:
CLASS_N = 40                                  # points per class in a 5-fold iris train fold
ceil_pool = OL.filter(pl.col('k').is_in([0.0, 8.0]))
CEIL = normalise(ceil_pool.filter(pl.col('n_out') > 0), ceil_pool.filter(pl.col('n_out') == 0))
doses = sorted(CEIL['n_out'].unique())
DC = {'toward': '#2f9e44', 'outward': '#adb5bd', 'random': '#9c36b5'}

if max(doses) < 20:
    print('No ceiling rows yet -- run:  python run_experiment.py --defect outlier --ceiling')
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.0), sharey=True)
    for ax, attack in zip(axes, ['hsj', 'deepfool']):
        for d in ['toward', 'outward', 'random']:
            sub = CEIL.filter((pl.col('attack') == attack) & (pl.col('direction') == d))
            m = [ci95(sub.filter(pl.col('n_out') == n)['norm'].to_list()) for n in doses]
            mu, hw = np.array([x[0] for x in m]), np.array([x[1] for x in m])
            pct = [n / CLASS_N * 100 for n in doses]
            ax.errorbar(pct, mu, yerr=hw, marker='o', ms=4, capsize=3, color=DC[d], label=d)
        ax.axhline(1.0, color='k', lw=0.8, ls=':')
        ax.axvline(100, color='k', lw=0.8, alpha=0.35)
        ax.text(101, ax.get_ylim()[1] * 0.995, 'class-size\nceiling', fontsize=7.5, va='top')
        ax.set_title(f'{attack}   (k=8)'); ax.set_xlabel('outliers injected  (% of target class)')

        # accuracy on a twin axis: spread rising WITH accuracy falling = contamination
        ax2 = ax.twinx()
        acc = [ci95(CEIL.filter((pl.col('attack') == attack) & (pl.col('direction') == 'toward') &
                                (pl.col('n_out') == n))['vacc'].to_list())[0] for n in doses]
        ax2.plot(pct, acc, ls='--', lw=1.2, color='#c92a2a', alpha=0.8)
        ax2.set_ylim(0.5, 1.02); ax2.grid(False)
        ax2.set_ylabel('test accuracy (toward)', color='#c92a2a', fontsize=9)
        ax2.tick_params(axis='y', colors='#c92a2a', labelsize=8)
    axes[0].set_ylabel('normalised spread  (1.0 = no effect)')
    axes[0].legend(loc='upper left', fontsize=8)
    fig.suptitle('Pushing the outlier dose past the class-size ceiling', y=1.03, fontsize=11)
    fig.tight_layout(); fig.savefig(PLOTS / 'f7_outlier_ceiling.png', bbox_inches='tight')
    plt.show()

## D2/D3 — fixed distance, rising count: threshold or dial?

The ceiling sweep held `k=8` and raised the count. This one holds the **distance** fixed at the
two near-field values and raises the count across the full range, with **`outward` at matched
count** as the control.

That control is what makes the result interpretable: without it, a compression at `k=2` is
indistinguishable from "60 extra training points fit a better model, so everything tightens".
`outward` adds the same number of points, the same distance out, in the opposite direction.

In [ ]:
dose_k = [2.0, 4.0]
DOSE = normalise(OL.filter((pl.col('n_out') > 0) & (pl.col('k').is_in(dose_k))),
                 OL.filter(pl.col('n_out') == 0))
ns = sorted(DOSE['n_out'].unique())

if len(ns) < 4:
    print('No fixed-distance dose rows yet -- run:  '
          'python run_experiment.py --defect outlier --dose')
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for ax, k in zip(axes, dose_k):
        for d, style in [('toward', '-'), ('outward', '--')]:
            for attack in ['deepfool', 'hsj']:
                sub = DOSE.filter((pl.col('k') == k) & (pl.col('direction') == d) &
                                  (pl.col('attack') == attack))
                m = [ci95(sub.filter(pl.col('n_out') == n)['norm'].to_list()) for n in ns]
                mu, hw = np.array([x[0] for x in m]), np.array([x[1] for x in m])
                ax.errorbar([n / CLASS_N * 100 for n in ns], mu, yerr=hw, ls=style, marker='o',
                            ms=3.5, capsize=2.5, color=C[attack],
                            alpha=1.0 if d == 'toward' else 0.35,
                            label=f'{attack} {d}' + (' (control)' if d == 'outward' else ''))
        ax.axhline(1.0, color='k', lw=0.8, ls=':')
        ax.set_xscale('log')
        ax.set_xlabel('outliers injected  (% of target class, log scale)')
        ax.set_title(f'k={k:.0f}   ({ {2.0: "55%", 4.0: "109%"}[k] } of the centroid gap)')
    axes[0].set_ylim(0.78, 1.22)
    # This signal runs DOWNWARD. Shading the compression half-plane is a guard against the
    # one-sided "is it above 1.0" read that caused H7 to be misreported as flat.
    for ax in axes:
        ax.axhspan(0.78, 1.0, color='#4c6ef5', alpha=0.06)
        ax.text(0.985, 0.985, 'compression', transform=ax.transAxes, ha='right', va='top',
                fontsize=8, color='#4c6ef5')
    axes[0].set_ylabel('normalised spread  (1.0 = no effect)')
    axes[0].legend(loc='upper left', fontsize=7.5, ncol=2)
    fig.suptitle('Fixed distance, rising count: the near-field effect is a threshold, not a dial',
                 y=1.03, fontsize=11)
    fig.tight_layout(); fig.savefig(PLOTS / 'f8_fixed_distance_dose.png', bbox_inches='tight')
    plt.show()

---
### Figures written to `plots/`

| file | shows |
|---|---|
| `f1_protocol_accuracy.png` | **the headline** — accuracy rises before-split, collapses train-only |
| `f2_coverage_gap_spread.png` | spread signal survives; the HSJ bias-0.9 drop is protocol-caused |
| `f4_outlier_model_comparison.png` | H6 refuted, MLP against all four previously-tested models |
| `h7_outlier_dose_response.png` | null on true outliers; DeepFool compresses on near-field contamination |
| `f7_outlier_ceiling.png` | dose pushed past the class-size ceiling (needs `--ceiling` rows) |